# Plot Loss History

This notebook reads the training `loss_history.csv` file and plots train and validation loss by epoch.

In [ ]:
from pathlib import Path
import csv

import matplotlib.pyplot as plt

In [ ]:
loss_history_path = Path("../checkpoints/quadra_fine_tune_v2/loss_history.csv")

if not loss_history_path.exists():
    raise FileNotFoundError(f"Could not find loss history CSV: {loss_history_path.resolve()}")

epochs = []
train_losses = []
val_losses = []
is_best_flags = []

with loss_history_path.open(newline="") as csv_file:
    reader = csv.DictReader(csv_file)
    for row in reader:
        epochs.append(int(row["epoch"]))
        train_losses.append(float(row["train_loss"]))
        val_losses.append(float(row["val_loss"]))
        is_best_flags.append(row["is_best"].strip().lower() == "true")

if not epochs:
    raise ValueError(f"Loss history CSV is empty: {loss_history_path.resolve()}")

best_epoch = None
best_val_loss = None
for epoch, val_loss, is_best in zip(epochs, val_losses, is_best_flags):
    if is_best:
        best_epoch = epoch
        best_val_loss = val_loss

if best_epoch is None:
    best_index = min(range(len(val_losses)), key=val_losses.__getitem__)
    best_epoch = epochs[best_index]
    best_val_loss = val_losses[best_index]

print(f"Loaded {len(epochs)} epochs from {loss_history_path.resolve()}")
print(f"Best validation loss: {best_val_loss:.6f} at epoch {best_epoch}")

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(epochs, train_losses, marker="o", linewidth=2, label="Train loss")
plt.plot(epochs, val_losses, marker="s", linewidth=2, label="Validation loss")
plt.scatter([best_epoch], [best_val_loss], color="red", s=80, label=f"Best val (epoch {best_epoch})")

plt.title("Training and Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.grid(True, linestyle="--", alpha=0.4)
plt.legend()
plt.tight_layout()
plt.show()